## 1. Load NHANES and score 3-year risk

Load the analysis file from `preprocessing.py` and add the CKD-PC 3-year risk of a ≥40% eGFR decline or kidney failure for every participant. Each person is scored with their own stratum (eGFR ≥60 or <60, with or without diabetes).

In [ ]:
import pandas as pd
from egfr40 import add_egfr40_risk_col

# Analysis-ready file from preprocessing.py
df = pd.read_parquet("../data/processed/nhanes_egfr40.parquet")

# Score every participant; column names already match the defaults
df = add_egfr40_risk_col(df)

# Label each participant's stratum for later summaries
df["stratum"] = (
    df["dm"].map({0: "No DM", 1: "DM"})
    + ", eGFR "
    + df["eGFR"].ge(60).map({True: ">=60", False: "<60"})
)

print(f"{len(df):,} participants, {df['egfr40_3year'].isna().sum()} missing scores")
df.groupby("stratum")["egfr40_3year"].describe().round(3)

## 1b. One risk score per participant

Every NHANES participant (one row per `SEQN`) now has an `egfr40_3year` score.
Show the first rows, then the ten highest-risk participants with the inputs that
drove their scores.

In [ ]:
# Columns worth seeing next to each participant's score
cols = ["SEQN", "stratum", "age", "sex", "eGFR", "uACR", "sbp", "bp_meds",
        "hf", "chd", "smoking", "bmi", "hba1c", "egfr40_3year"]

# First rows: one score per participant
display(df[cols].head(10).round(3))

# Highest-risk participants
display(df.sort_values("egfr40_3year", ascending=False)[cols].head(10).round(3))

## 2. Survey-weighted risk by stratum

NHANES oversamples some groups, so raw sample summaries don't describe the US population. Each participant's MEC exam weight (`WTMECPRP`) says how many US adults they represent. Weighting gives:

- **% of US adults** in each stratum
- **Weighted mean and median** 3-year risk
- **% at ≥10% risk**, a simple high-risk cutoff

These are complete-case estimates: weights are not re-adjusted for participants dropped for missing data, so treat population totals as approximate.

In [ ]:
import numpy as np


def weighted_quantile(values, weights, q):
    """Quantile q (0-1) of values, where each value counts by its weight."""
    order = np.argsort(values)
    v, w = np.asarray(values)[order], np.asarray(weights)[order]
    cum = np.cumsum(w) / w.sum()  # running share of total weight
    return v[np.searchsorted(cum, q)]


def weighted_summary(g):
    w, r = g["WTMECPRP"], g["egfr40_3year"]
    return pd.Series({
        "n": len(g),
        "us_adults_millions": w.sum() / 1e6,
        "mean_risk": np.average(r, weights=w),
        "median_risk": weighted_quantile(r, w, 0.5),
        "p90_risk": weighted_quantile(r, w, 0.9),
        "pct_risk_ge_10": np.average(r >= 0.10, weights=w) * 100,
    })


# Drop the few participants with zero weight (not examined)
dfw = df[df["WTMECPRP"] > 0]

by_stratum = dfw.groupby("stratum").apply(weighted_summary, include_groups=False)
overall = weighted_summary(dfw).rename("All adults")
summary = pd.concat([by_stratum, overall.to_frame().T])
summary["pct_us_adults"] = summary["us_adults_millions"] / overall["us_adults_millions"] * 100

summary.round({"us_adults_millions": 1, "mean_risk": 3, "median_risk": 3,
               "p90_risk": 3, "pct_risk_ge_10": 1, "pct_us_adults": 1})

## 3. Check real participants against the online calculator

The [ckdpcrisk.org/gfrdecline40](https://ckdpcrisk.org/gfrdecline40/) calculator only
accepts dropdown values (age in steps of 5, eGFR in steps of 5, and so on), while
NHANES values are continuous. So no participant can be entered exactly.

Instead, snap each participant's inputs to the nearest dropdown option and keep the
five participants whose inputs change the least. For each one, the package score on
the snapped inputs should match the calculator exactly, and should be close to the
participant's real score.

Check the dropdowns on the site and edit `GRID` if any options differ.

In [ ]:
import numpy as np
from egfr40 import egfr40_risk

# Dropdown options on the calculator (age and eGFR confirmed; edit others to match)
GRID = {
    "age":   np.arange(20, 81, 5),       # 20, 25, ..., 80
    "eGFR":  np.arange(15, 151, 5),      # steps of 5
    "uACR":  np.array([5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200, 300, 500, 1000]),
    "sbp":   np.arange(90, 201, 10),
    "bmi":   np.arange(15, 51, 1),
    "hba1c": np.arange(4, 15.5, 0.5),
}


def snap(x, grid):
    """Nearest dropdown value."""
    return grid[np.abs(grid - x).argmin()]


def snapped_row(r):
    """Snap one participant's inputs; HbA1c only matters with diabetes."""
    s = {k: snap(r[k], g) for k, g in GRID.items() if k != "hba1c" or r["dm"] == 1}
    # total relative change between real and snapped inputs
    s["snap_error"] = sum(abs(r[k] - s[k]) / max(abs(r[k]), 1) for k in s)
    return s


# Complete rows only, ranked by how little snapping changes them
cand = df.dropna(subset=["egfr40_3year"]).copy()
snaps = cand.apply(snapped_row, axis=1, result_type="expand")
cand = cand.join(snaps.add_prefix("site_"))
best = cand.nsmallest(5, "site_snap_error")

best[["SEQN", "stratum", "age", "site_age", "eGFR", "site_eGFR", "uACR", "site_uACR",
      "egfr40_3year", "site_snap_error"]].round(3)

### What to enter on the site

For each participant, the fields below are listed top to bottom in the same order and
with the same labels as the calculator. **DM Medication** and **HbA1c** appear only
when Diabetes is Yes, as on the site.

`-> site should show` is the package score on these exact inputs; the calculator should
display the same value. The real NHANES score is shown for comparison.

In [ ]:
SITE_ORDER = ["Diabetes", "Age", "Gender", "eGFR", "Urine Albumin to Creatinine", "SBP",
              "Antihypertensive Medication Use", "HF", "CHD", "Afib", "BMI",
              "Smoking History", "DM Medication", "HbA1c"]

def yn(x):
    return "Yes" if int(x) == 1 else "No"

for _, r in best.iterrows():
    dm = r.dm == 1
    entries = {
        "Diabetes": yn(r.dm),
        "Age": f"{r.site_age:.0f}",
        "Gender": r.sex.capitalize(),
        "eGFR": f"{r.site_eGFR:.0f}",
        "Urine Albumin to Creatinine": f"{r.site_uACR:.0f}",
        "SBP": f"{r.site_sbp:.0f}",
        "Antihypertensive Medication Use": yn(r.bp_meds),
        "HF": yn(r.hf),
        "CHD": yn(r.chd),
        "Afib": "No",
        "BMI": f"{r.site_bmi:.0f}",
        "Smoking History": {"never": "Never Smoked", "former": "Former Smoker",
                            "current": "Current Smoker"}[r.smoking],
    }
    if dm:  # the site only shows these two fields when Diabetes = Yes
        entries["DM Medication"] = ("Insulin use" if r.insulin == 1
                                    else "Oral Meds Only" if r.oral_dm == 1 else "No")
        entries["HbA1c"] = f"{r.site_hba1c:g}"

    site_risk = egfr40_risk(
        age=r.site_age, male=int(r.sex == "male"), egfr=r.site_eGFR, uacr=r.site_uACR,
        sbp=r.site_sbp, bp_meds=int(r.bp_meds), hf=int(r.hf), chd=int(r.chd), afib=0,
        smoking=r.smoking, bmi=r.site_bmi, dm=dm,
        hba1c=r.site_hba1c if dm else None,
        oral_dm=int(r.oral_dm), insulin=int(r.insulin),
    )

    print(f"SEQN {int(r.SEQN)}")
    for field in SITE_ORDER:
        if field in entries:
            print(f"  {field:<33} {entries[field]}")
    print(f"  {'-> site should show':<33} {site_risk:.1%}   (real NHANES: {r.egfr40_3year:.1%})\n")